# Propwash — BEMT propeller laboratory

An interactive blade-element-momentum propeller lab with a 3-D visualiser,
built to be run on a Google Colab GPU runtime.

**What it does.** You change a propeller — pitch, diameter, blade count, section,
air density — and it solves the aerodynamics and tells you two things: **how fast
the propeller ends up spinning**, and **how much thrust it makes**. The 3-D blade
is coloured by whatever solver field you pick and spins at a rate proportional to
the solved RPM, so the coupling is visible rather than just tabulated.

**Why speed is an output.** A propeller has no RPM of its own. It has a torque
demand that rises roughly as RPM², and a motor has a torque supply that falls
towards its no-load speed. Where the two cross is where it runs. Coarsen the
pitch and the crossing moves left — the propeller *slows down* and pulls more
current. That is the whole story this notebook is built around.

**Runtime.** Everything works on a CPU runtime. For the GPU parts pick
*Runtime → Change runtime type → T4 GPU* before running the setup cell.

---
## 1. Setup

Installs only what is missing, then reports what this particular host can do.
Safe to re-run.

In [ ]:
# Get the code onto the runtime.  Pick a GPU runtime BEFORE running this --
# changing the runtime type restarts the VM and wipes /content.
import pathlib, subprocess, sys

REPO = "https://github.com/abu-infidel/Python-based-BEMT-visualizer-calculator.git"
DEST = pathlib.Path("/content/propwash")


def find_package():
    """Locate propwash whether it was cloned, uploaded, or is already here."""
    for base in (DEST, pathlib.Path("."), pathlib.Path("..")):
        if (base / "propwash" / "__init__.py").exists():
            return base.resolve()
    hits = list(pathlib.Path("/content").rglob("propwash/__init__.py")) \
        if pathlib.Path("/content").exists() else []
    return hits[0].parent.parent if hits else None


root = find_package()
if root is None and pathlib.Path("/content").exists():      # on Colab: clone it
    result = subprocess.run(["git", "clone", "--depth", "1", REPO, str(DEST)],
                            capture_output=True, text=True)
    print(result.stderr.strip() or "cloned")
    root = find_package()

if root is None:
    raise SystemExit("propwash not found -- see docs/COLAB.md")

sys.path.insert(0, str(root))
print("propwash root:", root)

import propwash.colab as pc
env = pc.setup()          # installs numba / plotly / ipywidgets / cupy as needed

### What did we get?

`nvidia-smi` tells you the card; the backend list tells you which compute paths
actually import and initialise. A backend that is missing says *why*, and the
solver quietly uses the next one down — nothing here fails because a GPU is
absent.

In [ ]:
print(env.report())

---
## 2. One operating point

Before any GUI: the plain API. Pick a propeller, pick an RPM and an airspeed,
get a full solution including the spanwise state of every blade element.

In [ ]:
from propwash import get_preset, OperatingPoint, PropellerSolver, list_presets

print(list_presets())

prop = get_preset("APC 10x5 (sport)")
solver = PropellerSolver(prop)
result = solver.solve(OperatingPoint(rpm=9000, v_inf=12.0))

print()
print(prop.describe())
for key, value in result.summary().items():
    print(f"  {key:<20} {value:,.5g}")

---
## 3. The interactive lab

The main event. Controls on the left, tabs on the right.

* **Flight condition → Motor-matched** is on by default, so the *Shaft speed*
  slider is read-only — it shows the solved answer. Switch to *Fixed RPM* to
  drive it yourself.
* **Propeller → Collective** is the single most instructive slider. Add pitch:
  thrust rises, RPM falls, current climbs. Take pitch away and it spins up.
* **View → Colour by** maps any solver field onto the blade. `Thrust loading`
  shows where the work happens; `Angle of attack` uses a diverging scale so you
  can see the stalled root.
* **View → Spin** animates at a rate proportional to the solved RPM.
* The **Benchmark** tab is where you find this host's limits.

In [ ]:
lab = pc.launch(preset="APC 10x5 (sport)", motor="Sport 2820 kv1000")

---
## 4. What the solver is doing

BEMT balances two descriptions of the same blade element: the **blade-element**
view (a little wing with a lift and a drag) and the **momentum** view (an
annulus of the actuator disk accelerating air). They agree at exactly one inflow
angle φ, which is the root of

$$\mathcal{R}(\phi) = \Omega r\,(4F\sin^2\phi - \sigma C_n)
                     - V\,(\sigma C_t + 4F\sin\phi\cos\phi)$$

This particular form was chosen for three reasons: it contains no division, so
nothing blows up; it stays finite at *V* = 0, which is the static-thrust case
everyone looks at first; and it provably changes sign on (0, π/2), so bracketed
bisection always converges. Fixed iteration count, no data-dependent branching —
which is also exactly what makes it a good GPU kernel.

The chart below is the residual for one blade element. The root is where the
physics balances.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from propwash.bemt.core import residual_phi, _batch_context, SolverOptions
from propwash.atmosphere import SEA_LEVEL
from propwash.units import rpm_to_rad_s
from propwash.viz.palette import series_color, theme
from propwash.viz.plots import style_axes

opts = SolverOptions()
st = prop.discretize(opts.n_elements)
ctx = _batch_context(st, np.array([rpm_to_rad_s(9000.0)]), np.array([12.0]),
                     SEA_LEVEL, opts.flags(), st.polar_tables())

phi = np.linspace(1e-4, np.pi/2 - 1e-4, 900)
station = 3 * st.n // 4                     # the 75% station
R = np.array([residual_phi(np.full((1, st.n), p), ctx)[0, station] for p in phi])

fig, ax = plt.subplots(figsize=(7.5, 3.6), facecolor=theme()["surface"])
ax.plot(np.degrees(phi), R, color=series_color(0), linewidth=1.8)
ax.axhline(0.0, color=theme()["axis"], linewidth=0.9)
root = np.degrees(result.phi[station])
ax.plot([root], [0.0], "o", markersize=8, color=theme()["text"],
        markeredgecolor=theme()["surface"], markeredgewidth=1.6, zorder=5)
ax.annotate(f"solution  phi = {root:.2f} deg", xy=(root, 0.0), xytext=(12, 22),
            textcoords="offset points", fontsize=9)
ax.set_xlabel("inflow angle phi [deg]")
ax.set_ylabel("residual")
ax.set_title(f"BEMT residual at r/R = {st.x[station]:.2f}", fontsize=10, loc="left")
style_axes(ax)
plt.show()

### Spanwise state

Six single-scale panels. Read them together: the angle of attack falls outboard
because the blade is twisted; lift stays roughly constant across the working
span; thrust loading peaks around 80% and collapses at the tip, which is the
Prandtl tip-loss factor doing its job.

In [ ]:
from propwash.viz.plots import spanwise_figure
spanwise_figure(result, solver.stations)
plt.show()

---
## 5. The propeller chart

Sweeping advance ratio *J* = V/(nD) at fixed RPM collapses the performance onto
curves that do not depend on size or speed separately. Efficiency is reported as
zero past the zero-thrust *J*: beyond that point the propeller is a brake and
T·V/P stops meaning anything.

In [ ]:
from propwash.bemt.sweep import j_sweep
from propwash.viz.plots import j_sweep_figure

sweep = j_sweep(prop, rpm=9000, j_max=1.2, n=120)
j_sweep_figure(sweep)
plt.show()

best = sweep.best("efficiency")
print(f"peak efficiency {best['efficiency']:.3f} at J = {best['j']:.3f} "
      f"(V = {best['v_inf']:.1f} m/s, thrust {best['thrust']:.2f} N)")

---
## 6. Where does it actually spin?

Two curves, one scale. The propeller's torque demand rises with RPM; the
motor's supply falls (flat at the top where the ESC current limit bites). They
cross once. That crossing is the answer to "how fast does it spin".

Change the pitch or the cell count and watch the crossing move.

In [ ]:
from propwash.motor import get_motor
from propwash.bemt.sweep import match_operating_point
from propwash.viz.plots import matching_figure

motor = get_motor("Sport 2820 kv1000")
rpm = np.linspace(300, motor.no_load_rpm() * 1.03, 90)

match = match_operating_point(prop, motor)
matching_figure(rpm, solver.solve_many(rpm, np.zeros_like(rpm))["torque"],
                motor.shaft_torque(rpm), match.rpm)
plt.show()
print(match.describe())

In [ ]:
# One variable at a time, everything else held.  This is the table the GUI
# is drawing continuously.
from dataclasses import replace
import math

rows = []
def probe(tag, geometry=prop, m=motor, v=0.0):
    mp = match_operating_point(geometry, m, v_inf=v)
    rows.append((tag, mp.rpm, mp.thrust, mp.current))

probe("baseline")
probe("collective +4 deg", replace(prop, pitch_offset=math.radians(4)))
probe("collective -4 deg", replace(prop, pitch_offset=math.radians(-4)))
probe("3 blades", replace(prop, n_blades=3))
probe("12 in diameter", replace(prop, radius=12 * 0.0254 / 2))
probe("4S pack", m=motor.with_cells(4))
probe("15 m/s forward", v=15.0)

print(f"{'change':<22}{'rpm':>9}{'thrust N':>11}{'amps':>8}")
for tag, r, t, a in rows:
    print(f"{tag:<22}{r:>9,.0f}{t:>11.2f}{a:>8.1f}")

---
## 7. Parameter maps — the part that wants a GPU

A single BEMT solve is microseconds. The interesting questions are maps, and a
map is thousands to millions of solves. This one is a collective-pitch × RPM
grid: every cell is a full converged solution on every blade element.

In [ ]:
from propwash.bemt.sweep import pitch_rpm_map
from propwash.viz.plots import map_figure
import time

pitch = np.linspace(-8, 14, 45)
rpms  = np.linspace(1500, 14000, 60)

t0 = time.perf_counter()
grid = pitch_rpm_map(prop, pitch, rpms, v_inf=0.0)
dt = time.perf_counter() - t0
print(f"{grid.shape[0] * grid.shape[1]:,} operating points in {dt:.2f} s")

map_figure(grid, "thrust", mark_best="thrust")
plt.show()
map_figure(grid, "figure_of_merit", mark_best="figure_of_merit")
plt.show()

---
## 8. How hard can you push this host?

Every backend solves the same grid and is diffed against the NumPy reference.
A GPU path that is a hundred times faster and half a percent wrong is a bug, so
correctness is reported next to speed.

On a Colab **T4** expect the CUDA backends to be one to two orders of magnitude
faster than the threaded CPU path; on a **CPU runtime** you will see NumPy and
Numba only, and everything still works.

In [ ]:
from propwash.accel.bench import benchmark

report = benchmark(prop, n_cases=16384, n_elements=64, verbose=False)
print(report.format())

In [ ]:
# Throughput against problem size: where does a GPU start to pay for itself?
from propwash.accel.bench import sizing_sweep
from propwash.viz.palette import series_color

scaling = sizing_sweep(sizes=(256, 1024, 4096, 16384, 65536),
                       geometry=prop, n_elements=48)
sizes = scaling.pop("sizes")

fig, ax = plt.subplots(figsize=(7.2, 4.0), facecolor=theme()["surface"])
for i, (name, rate) in enumerate(scaling.items()):
    ax.loglog(sizes[:len(rate)], rate, "o-", color=series_color(i), linewidth=1.8,
              markersize=6, label=name)
ax.set_xlabel("operating points per call")
ax.set_ylabel("operating points per second")
ax.set_title("Backend throughput vs problem size", fontsize=10, loc="left")
ax.legend(frameon=False, fontsize=9)
style_axes(ax)
plt.show()

---
## 9. The CUDA kernel

There are two GPU implementations and they are deliberately different in kind:

* **Numba CUDA** — compiled from the same scalar Python functions as the CPU
  kernel, so the two cannot drift apart.
* **Raw CUDA C** — hand written, compiled by CuPy at runtime, and the version
  you can read, profile with Nsight, or tune the block size of.

Both use one block per operating point with threads striding over blade
stations and a shared-memory tree reduction for thrust and torque. The bisection
runs a fixed number of iterations with no early exit, so every thread in a warp
retires together.

The cell below compiles the CUDA C with **NVRTC**, which needs no GPU at all —
so you can check the kernel on a CPU runtime before paying for a GPU one.

In [ ]:
from propwash.accel.nvrtc import check, find_library
from propwash.accel.kernels_raw import kernel_source

if find_library() is None:
    import subprocess, sys as _sys
    print("installing NVRTC (a compiler only -- no GPU needed) ...")
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                    "nvidia-cuda-nvrtc-cu12"], check=False)

check(("compute_75", "compute_80", "compute_90"))

In [ ]:
# The device code itself -- the bisection loop and the reduction.
src = kernel_source()
start = src.index("extern \"C\" __global__")
print(src[start:start + 2600])

---
## 10. Export

The blade goes out as OBJ and STL (open them in Blender, or print one), the
geometry as JSON, and the spanwise solution as CSV.

In [ ]:
from propwash.mesh import build_propeller_mesh
from propwash.bemt.solver import spanwise_frame
from pathlib import Path

out = Path("exports"); out.mkdir(exist_ok=True)
mesh = build_propeller_mesh(prop)
(out / "blade.obj").write_text(mesh.to_obj())
(out / "blade.stl").write_bytes(mesh.to_stl())
prop.save(out / "blade.json")
spanwise_frame(result, solver.stations).to_csv(out / "spanwise.csv", index=False)

print(mesh.describe())
for p in sorted(out.iterdir()):
    print(f"  {p.name:<18} {p.stat().st_size/1024:8.1f} kB")

---
## Where the numbers come from, and what they are worth

BEMT is a strip theory: it treats each radial station as an isolated 2-D aerofoil
and couples it to 1-D momentum on an annulus. That is an excellent trade for a
tool you want to run a million times, and it is wrong in known ways.

Against published static data for an APC 10x5 this model lands within roughly
10%, under-predicting, which is the normal direction for BEMT — it does not model
hub and spinner blockage, radial flow, or the real blade's exact geometry.

Specific limits worth knowing:

* Only the **normal thrusting branch** is bracketed. Windmilling and
  propeller-brake states have their root outside (0, π/2); those elements are
  reported as non-converged rather than silently returned as nonsense. The
  status line tells you when that happens.
* The bundled aerofoil polars are **parametrised models** fitted to published
  section characteristics, not digitised wind-tunnel data. Load your own XFOIL
  output with `propwash.airfoil.load_polar_file` for serious work.
* Deep post-stall uses **Viterna–Corrigan** extrapolation, which exists to keep
  the solver convergent, not to be accurate at 60° angle of attack.
* There is no unsteadiness, no blade flex, no non-axial inflow.